In [1]:
# Built-in Imports
from pathlib import Path

# Dependancy Imports
import scanpy as sc
import pandas as pd
from dotenv import dotenv_values

In [2]:
# Project-wide Constants
CONFIG = dotenv_values("config.env") 
DATADIR = Path(CONFIG['DATADIR'])
N_HVGS = 4500

# next paths will be derived
DATASET = DATADIR / f'2b_clean/annconcat.h5ad' 
OUTFILE_CONTROL = DATADIR / f"3_solo/solo_control_4500.csv"

### Let's get all control samples

In [3]:
adata = sc.read_h5ad(DATASET)
adata = adata[adata.obs.IsControl == True]

In [4]:
adata 

View of AnnData object with n_obs × n_vars = 1165539 × 56939
    obs: 'CellID', 'Dataset', 'Sample', 'total_counts', 'n_genes', 'pct_mito', 'Author', 'Source', 'Data_source_link', 'GEO', 'Publication_Link', 'Suspension', 'Technology', 'Author_Sex_harmonized', 'Strain', 'Diet', 'Treatment', 'Age', 'Mean_age_in_weeks', 'IsControl', 'cluster_id_label', 'supertype_id_label', 'subclass_id_label', 'class_id_label', 'C66_named', 'C185_named', 'C286_named', 'C465_named'
    var: 'gene_ids', 'IsMouseTF', 'IsHumanTFOrtholog', 'IsSourceGene', 'IsTargetGene', 'IsHvgExcludedGene', 'IsAnnotationExcludedGene', 'IsPriorityGene', 'IsTopGene'

### and get the HVGs

In [5]:
hvg_df = sc.pp.highly_variable_genes(adata, 
                                    inplace=False, 
                                    n_top_genes=adata.shape[1],
                                    batch_key='Sample',
                                    flavor='seurat_v3',
                                    span=1)

# Sort the DataFrame by the 'highly_variable_rank' column
hvg_df = hvg_df.sort_values('highly_variable_rank')

In [6]:
excluded_features = list(adata.var.loc[adata.var.IsHvgExcludedGene, :].index)

# Identify genes to exclude and set their rank to None
excluded_mask = hvg_df.index.isin(excluded_features)
hvg_df.loc[excluded_mask, 'highly_variable_rank'] = None

# Re-sort the DataFrame after modification
hvg_df = hvg_df.sort_values('highly_variable_rank')

In [7]:
hvg_df.head(N_HVGS)

,means,variances,gene_name,highly_variable_nbatches,highly_variable_rank,variances_norm,highly_variable
Avp,2.877416,3945.718447,Avp,201,5.0,16.220773,True
Sst,1.121524,205.328537,Sst,201,6.0,14.067975,True
Ptgds,4.738150,927.030549,Ptgds,201,20.0,10.278281,True
Gal,0.922548,130.414495,Gal,201,24.0,8.405100,True
Cartpt,0.916857,109.510043,Cartpt,201,24.0,9.067876,True
...,...,...,...,...,...,...,...
Clybl,0.268673,0.623040,Clybl,201,7850.0,1.070805,True
Gm56790,0.008720,0.011095,Gm56790,201,7851.0,0.924748,True
Prrg4,0.029633,0.039146,Prrg4,201,7853.0,1.006258,True
Slc6a12,0.003905,0.007610,Slc6a12,201,7854.0,1.033709,True


save top N_HVGS

In [9]:
hvg_df.head(N_HVGS).gene_name.to_csv(OUTFILE_CONTROL, index=False, header=False)